# AgriShield

End-to-end soil-risk pipeline: harmonise surveys, train a classifier, query live satellite + climate for a new site.

```
LUCAS + WoSIS  ->  data/agrishield_training.csv  ->  XGBoost (tuned)
New lat/lon    ->  GEE (Sentinel-2, WorldClim, static soil)  ->  risk report
```

**Hard rule this notebook follows:** every column in `FEATURE_COLUMNS` (agrishield/config.py) must be obtainable live, from lat/lon alone, with no lab test. Lab-only chemistry (OC, N, P, K, EC, CEC, bulk density) is the TARGET, never a feature.

Run cells top to bottom. Enrichment (2b) auto-skips itself if the training table already has satellite/climate columns, so re-running this notebook top-to-bottom after the first full run will NOT repeat the slow GEE pass or touch your data.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agrishield.config import TRAINING_CSV, FEATURE_COLUMNS, EXAMPLE_SITE
from agrishield.dataset import build_training_csv
from agrishield.climate import enrich_training_csv
from agrishield.model import train, save_model, predict_proba
from agrishield.inference import live_features, live_model_inputs

pd.set_option("display.max_columns", 40)
print("root:", ROOT)
print("training table exists:", TRAINING_CSV.exists())

root: d:\Jupyter Lab\Agrishield AI
training table exists: True


## 2. Base training table -- LUCAS + WoSIS only

No APIs yet, pure local files. Fast (seconds, not minutes).

This cell NEVER rebuilds if `TRAINING_CSV` already exists on disk -- it just loads it. `build_training_csv()` only runs the first time, when there's nothing there yet. This is deliberate: `build_training_csv()` only knows how to produce the raw 38-column merge, so calling it again on an already-enriched file would silently throw away every satellite/climate column you already paid GEE quota for.

`sample_year` is kept as a lookup key for step 2b -- it decides which satellite image to fetch for each row. It is never a model feature; check `FEATURE_COLUMNS` above if you want to confirm.

In [2]:
if TRAINING_CSV.exists():
    df = pd.read_csv(TRAINING_CSV, low_memory=False)
    print("loaded existing training table (NOT rebuilt) -- rows:", len(df), " cols:", df.shape[1])
else:
    df = build_training_csv()
    print("built fresh training table -- rows:", len(df), " cols:", df.shape[1])
print(df["source"].value_counts())
display(df.head())

loaded existing training table (NOT rebuilt) -- rows: 250578  cols: 53
source
wosis         195011
lucas_2009     19791
lucas_2018     18983
lucas_2015     16793
Name: count, dtype: int64


,source,sample_id,latitude,longitude,country,continent,sample_year,ph_h2o,ph_cacl2,oc_gkg,oc_20_30_gkg,n_gkg,p_mgkg,k_mgkg,ec,caco3,caco3_20_30,ox_al,ox_fe,clay_pct,...,cec_ph7,totc_gkg,acidic,low_oc,soil_stress,tmean_c,temp_seasonality,precip_mm,precip_seasonality,slope_deg,B2,B3,B4,B5,B6,B7,B8,B11,B12,ndvi
0,lucas_2018,47862690,47.150238,16.134212,AT,Europe,2018.0,4.81,4.1,12.4,NaN,1.1,NaN,101.9,8.73,3.0,NaN,NaN,NaN,24.0,...,NaN,NaN,1.0,0.0,1.0,9.1,7385.0,757.0,37.0,16.680049,239.000000,408.000,276.0,605.000000,1707.000000,2077.000000,2142.00,1017.000,486.000000,0.771712
1,lucas_2018,47882704,47.274272,16.175359,AT,Europe,2018.0,4.93,4.1,16.7,NaN,1.3,NaN,51.2,5.06,1.0,NaN,NaN,NaN,20.0,...,NaN,NaN,1.0,0.0,1.0,8.8,7352.0,744.0,37.0,4.197045,253.000000,437.000,353.0,812.000000,2307.000000,2837.000000,2698.00,1455.000,715.000000,0.768600
2,lucas_2018,47982688,47.123260,16.289693,AT,Europe,2018.0,4.85,4.1,47.5,NaN,3.1,12.3,114.8,12.53,1.0,NaN,NaN,NaN,23.0,...,NaN,NaN,1.0,0.0,1.0,9.3,7360.0,727.0,36.0,3.946203,223.000000,440.000,317.0,681.000000,2095.000000,2471.000000,2596.00,1458.000,703.000000,0.782355
3,lucas_2018,48022702,47.245693,16.357506,AT,Europe,2018.0,5.80,5.5,28.1,NaN,2.0,NaN,165.8,21.10,3.0,NaN,NaN,NaN,26.0,...,NaN,NaN,0.0,0.0,0.0,9.1,7324.0,701.0,37.0,9.195145,355.882353,548.200,487.0,793.812500,1985.383333,2409.666667,2602.75,1460.700,828.958333,0.684764
4,lucas_2018,48062708,47.296372,16.416782,AT,Europe,2018.0,6.48,6.1,19.4,NaN,2.2,NaN,42.1,10.89,2.0,NaN,NaN,NaN,25.0,...,NaN,NaN,0.0,0.0,0.0,9.0,7321.0,687.0,37.0,10.184996,286.190476,450.725,305.0,741.153846,2025.750000,2366.050000,2626.50,1376.625,725.700000,0.791915


### 2a. Know your date coverage before spending GEE quota

Sentinel-2 (the satellite you're querying in 2b) only exists from 2015 onward. Rows with an older or missing `sample_year` will end up with satellite columns as `NaN` -- expected, not a bug. WorldClim and static soil/elevation don't depend on date, so they'll still fill in for almost every row.

In [3]:
has_year = df["sample_year"].notna()
print("missing sample_year:", df["sample_year"].isna().sum(),
      f"({df['sample_year'].isna().mean()*100:.1f}%)")
print("year < 2015 (pre-Sentinel-2):", (df.loc[has_year, "sample_year"] < 2015).sum())
print("year >= 2015 (real spectral data possible):", (df.loc[has_year, "sample_year"] >= 2015).sum())

missing sample_year: 94427 (37.7%)
year < 2015 (pre-Sentinel-2): 120226
year >= 2015 (real spectral data possible): 35925


## 2b. Attach satellite + climate + soil columns

Three things get attached to the SAME rows as new columns:
- **Sentinel-2 bands + NDVI** -- batched per `sample_year` (one composite image per year, not one API call per row). Only fills for rows with `sample_year >= 2016` (rows before Sentinel-2 launched are skipped entirely -- not a failure, just physically impossible).
- **Static soil texture + elevation** -- from OpenLandMap/SRTM. Fills for virtually every row with coordinates, and OVERWRITES the LUCAS lab-measured `clay_pct`/`sand_pct`/`silt_pct`/`elevation_m` so training and live inference read from the identical source.
- **WorldClim** -- climatology, fills for virtually every row.

This cell auto-detects whether enrichment already ran (checks for `B2`/`tmean_c` with real values) and skips the slow GEE pass if so -- safe to re-run top-to-bottom any time.

In [4]:
probe_cols = ["B2", "tmean_c"]
already_enriched = all(c in df.columns for c in probe_cols) and df[probe_cols].notna().any().all()

if already_enriched:
    print("training table already enriched -- skipping GEE calls")
else:
    df = enrich_training_csv(batch_size=400, max_rows=None)
    print("enrichment complete -- rows:", len(df), " cols:", df.shape[1])

training table already enriched -- skipping GEE calls


## 3. Train

Loads from disk (not the in-memory `df` above) so this cell works even if you restarted the kernel after the overnight run finished. This is the production model -- tuned XGBoost, trained on all rows (satellite-era and pre-satellite alike; non-satellite rows just get median-imputed band values). Hyperparameters live in `model.py`'s `_TUNED_XGB_PARAMS`; see section 7 below if you want to re-search them.

In [5]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("training on", len(df), "rows")
model, report = train(df)
print(report)
save_model(model)

cols = [c for c in FEATURE_COLUMNS if c in df.columns]
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=cols).sort_values(ascending=False)
print("\nfeature importances:")
print(importances)

training on 250578 rows
class balance (target = acidic):
acidic
0.0    0.694
1.0    0.306
              precision    recall  f1-score   support

         0.0      0.901     0.818     0.858     26652
         1.0      0.655     0.793     0.718     11604

    accuracy                          0.811     38256
   macro avg      0.778     0.806     0.788     38256
weighted avg      0.826     0.811     0.815     38256


feature importances:
clay_pct              0.157879
precip_mm             0.127062
tmean_c               0.079440
B12                   0.071682
B3                    0.068668
precip_seasonality    0.057847
temp_seasonality      0.052389
elevation_m           0.049306
sand_pct              0.038892
slope_deg             0.037147
silt_pct              0.035173
B5                    0.033541
B11                   0.031313
ndvi                  0.028469
B6                    0.026898
B7                    0.026817
B2                    0.026367
B8                    0.025734
B4 

### 3b. Optional comparison: satellite-era rows only (`sample_year >= 2015`)

Same model, trained on just the rows that have REAL (not imputed) spectral bands, so you can see whether accuracy actually improves on cleaner data or whether the extra pre-satellite rows were pulling their weight.

In [6]:
df_recent = df[df["sample_year"] >= 2015].copy()
print("rows with real satellite era data:", len(df_recent))
model_recent, report_recent = train(df_recent)
print("--- full dataset ---")
print(report)
print("--- satellite-era only ---")
print(report_recent)

rows with real satellite era data: 35925
class balance (target = acidic):
acidic
0.0    0.652
1.0    0.348
--- full dataset ---
              precision    recall  f1-score   support

         0.0      0.901     0.818     0.858     26652
         1.0      0.655     0.793     0.718     11604

    accuracy                          0.811     38256
   macro avg      0.778     0.806     0.788     38256
weighted avg      0.826     0.811     0.815     38256

--- satellite-era only ---
              precision    recall  f1-score   support

         0.0      0.881     0.885     0.883      4572
         1.0      0.796     0.789     0.793      2603

    accuracy                          0.850      7175
   macro avg      0.838     0.837     0.838      7175
weighted avg      0.850     0.850     0.850      7175



## 4. Live inference for a new site

This is what actually runs when a farmer taps Calculate -- one coordinate, live GEE + weather calls, no training data touched. `live_model_inputs` feeds the model; `live_features` wraps that plus current weather for a human-facing report.

In [7]:
site = EXAMPLE_SITE
report_data = live_features(site["latitude"], site["longitude"])
print(site["name"], site["latitude"], site["longitude"])
report_data

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Clayton, Victoria -37.91 145.13


{'latitude': -37.91,
 'longitude': 145.13,
 'model_inputs': {'B2': 296,
  'B3': 453.5,
  'B4': 452.5,
  'B5': 1122.5,
  'B6': 2000,
  'B7': 2296,
  'B8': 2312.5,
  'B11': 1541.5,
  'B12': 1168.5,
  'ndvi': 0.6726943850517273,
  'elevation_m': 101,
  'slope_deg': 0.3743423819541931,
  'clay_pct': 16,
  'sand_pct': 70,
  'silt_pct': 14.0,
  'tmean_c': 14.4,
  'temp_seasonality': 3660,
  'precip_mm': 836,
  'precip_seasonality': 17},
 'confidence': {'continent': 'Oceania',
  'tier': 'higher',
  'held_out_recall': 0.673,
  'note': "When Oceania was fully excluded from training and tested cold, the model still caught 67% of real acidic soil there. Treat this as a regional reliability signal, not the model's own confidence."},
 'weather_now': {'temp_c': 22.25,
  'humidity': 52,
  'pressure': 1024,
  'wind_speed': 2.44,
  'rain_1h_mm': 0,
  'description': 'overcast clouds'}}

In [8]:
inputs = live_model_inputs(site["latitude"], site["longitude"])
print(inputs)
predict_proba(model, inputs)

{'B2': 296, 'B3': 453.5, 'B4': 452.5, 'B5': 1122.5, 'B6': 2000, 'B7': 2296, 'B8': 2312.5, 'B11': 1541.5, 'B12': 1168.5, 'ndvi': 0.6726943850517273, 'elevation_m': 101, 'slope_deg': 0.3743423819541931, 'clay_pct': 16, 'sand_pct': 70, 'silt_pct': 14.0, 'tmean_c': 14.4, 'temp_seasonality': 3660, 'precip_mm': 836, 'precip_seasonality': 17}


{'predicted': 1,
 'probability': {0: 0.42549216747283936, 1: 0.5745078325271606}}

## 5. Diagnostics

Not part of the production path -- these cells check the training table and the model's behaviour before you trust it. Keep them; they're what catch a silently-broken enrichment run or a model that's leaning on geography instead of soil physics.

In [9]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("total rows:", len(df))

sentinel_cols = ["B2","B3","B4","B5","B6","B7","B8","B11","B12","ndvi"]
static_cols   = ["clay_pct","sand_pct","silt_pct","elevation_m","slope_deg"]
climate_cols  = ["tmean_c","precip_mm","temp_seasonality","precip_seasonality"]

print("\n--- coverage (fraction non-null) ---")
print(df[sentinel_cols + static_cols + climate_cols].notna().mean().round(3).to_string())

print("\nrows with at least one real sentinel band:", df[sentinel_cols].notna().any(axis=1).sum())
print("rows with sample_year >= 2016:", (df["sample_year"] >= 2016).sum())

total rows: 250578

--- coverage (fraction non-null) ---
B2                    0.076
B3                    0.076
B4                    0.076
B5                    0.076
B6                    0.076
B7                    0.076
B8                    0.076
B11                   0.076
B12                   0.076
ndvi                  0.076
clay_pct              0.993
sand_pct              0.993
silt_pct              0.997
elevation_m           0.960
slope_deg             0.960
tmean_c               0.998
precip_mm             0.998
temp_seasonality      0.998
precip_seasonality    0.998

rows with at least one real sentinel band: 19091
rows with sample_year >= 2016: 19091


In [10]:
sat_subset = df[df["B2"].notna()].copy()
print("rows with real satellite data:", len(sat_subset))

sat_cols = ["B2","B3","B4","B5","B6","B7","B8","B11","B12","ndvi"]

# WITH satellite bands
model_with, report_with = train(sat_subset)

# WITHOUT -- same rows, just drop the band columns before training
sat_subset_no_bands = sat_subset.drop(columns=[c for c in sat_cols if c in sat_subset.columns])
model_without, report_without = train(sat_subset_no_bands)

print("--- WITH satellite bands ---")
print(report_with)
print("--- WITHOUT satellite bands (same rows) ---")
print(report_without)

rows with real satellite data: 19091
class balance (target = acidic):
acidic
0.0    0.68
1.0    0.32
class balance (target = acidic):
acidic
0.0    0.68
1.0    0.32
--- WITH satellite bands ---
              precision    recall  f1-score   support

         0.0      0.895     0.906     0.901      2561
         1.0      0.804     0.784     0.794      1257

    accuracy                          0.866      3818
   macro avg      0.850     0.845     0.848      3818
weighted avg      0.865     0.866     0.866      3818

--- WITHOUT satellite bands (same rows) ---
              precision    recall  f1-score   support

         0.0      0.876     0.879     0.877      2561
         1.0      0.751     0.745     0.748      1257

    accuracy                          0.835      3818
   macro avg      0.813     0.812     0.813      3818
weighted avg      0.835     0.835     0.835      3818



In [11]:
print(df["continent"].value_counts(dropna=False))

continent
Europe              91927
Northern America    67179
Oceania             32485
Africa              28598
South America       23434
Asia                 6920
Antarctica             35
Name: count, dtype: int64


## 6. Continent generalization check

Trains on every continent but one, tests on the one left out. This is the honest number -- in-distribution accuracy (section 3) tells you how well the model fits data like what it trained on; this tells you how it does on a region it has genuinely never seen. Expect lower recall on the acidic class here than in section 3 -- that gap is the model leaning partly on regional climate/texture signature rather than pure soil physics, and it's the main thing more balanced regional data (Africa/Asia/South America) should improve.

In [12]:
from agrishield.model import continent_holdout_check
continent_holdout_check(df)

c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



--- trained WITHOUT Europe, tested ON Europe (n=74127) ---
              precision    recall  f1-score   support

         0.0      0.802     0.723     0.760     48952
         1.0      0.548     0.652     0.595     25175

    accuracy                          0.699     74127
   macro avg      0.675     0.687     0.678     74127
weighted avg      0.715     0.699     0.704     74127


--- trained WITHOUT Africa, tested ON Africa (n=23248) ---
              precision    recall  f1-score   support

         0.0      0.843     0.739     0.788     17687
         1.0      0.404     0.564     0.471      5561

    accuracy                          0.697     23248
   macro avg      0.624     0.651     0.629     23248
weighted avg      0.738     0.697     0.712     23248


--- trained WITHOUT Northern America, tested ON Northern America (n=57115) ---
              precision    recall  f1-score   support

         0.0      0.801     0.754     0.777     41643
         1.0      0.428     0.497    

## 7. Hyperparameter search (optional, slow)

30 candidates x 5-fold grouped CV = 150 full fits on the whole table -- this can take a long time on a laptop. Only run it when you actually want to re-tune (e.g. after adding a lot of new regional data, since the best settings for the current class/region balance may not be best for a different one). Scoring uses `fbeta` with `beta=1.5`, which deliberately favors recall on the acidic class over precision -- missing real acidic soil matters more here than a false alarm. If that trade-off should be different for your use case, change `beta` before running.

This cell only searches and prints results -- it does NOT update `model.py`. If a search finds something better, copy the printed `best params` into `model.py`'s `_TUNED_XGB_PARAMS` by hand, then use the next cell to confirm it's actually an improvement before trusting it.

In [13]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import make_scorer, fbeta_score
from scipy.stats import randint, uniform
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from agrishield.config import TARGET_COLUMN
from agrishield.model import available_features

cols = available_features(df)
work = df.dropna(subset=[TARGET_COLUMN]).copy()
X, y, groups = work[cols], work[TARGET_COLUMN], work["sample_id"]

neg, pos = (y == 0).sum(), (y == 1).sum()
base_scale = neg / pos

param_dist = {
    "clf__n_estimators": randint(150, 500),
    "clf__max_depth": randint(3, 8),
    "clf__learning_rate": uniform(0.03, 0.27),
    "clf__subsample": uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__min_child_weight": randint(1, 10),
    "clf__scale_pos_weight": uniform(base_scale * 0.7, base_scale * 0.6),
}

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", xgb.XGBClassifier(random_state=0, n_jobs=-1, eval_metric="logloss")),
])

recall_scorer = make_scorer(fbeta_score, beta=1.5)  # >1 favors recall over precision
cv = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    pipe, param_distributions=param_dist, n_iter=30,
    scoring=recall_scorer, cv=cv, n_jobs=-1, random_state=0, verbose=2,
)
search.fit(X, y, groups=groups)
print("best params:", search.best_params_)
print("best CV fbeta(1.5):", search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
best params: {'clf__colsample_bytree': np.float64(0.9584153550036308), 'clf__learning_rate': np.float64(0.2025086905824545), 'clf__max_depth': 6, 'clf__min_child_weight': 1, 'clf__n_estimators': 410, 'clf__scale_pos_weight': np.float64(2.9232096981525153), 'clf__subsample': np.float64(0.6464807638449578)}
best CV fbeta(1.5): 0.754125570265493


### 7b. Confirm a candidate is actually better before adopting it

Compares whatever is currently deployed in `model.py` against a candidate you fill in below. Leave `CANDIDATE_PARAMS` empty to just re-check the current default's numbers.

In [14]:
from agrishield.model import train, continent_holdout_check

CANDIDATE_PARAMS = {
    # paste the next search's "best params" here before running this cell.
    # leaving this empty just re-prints the current deployed default's numbers.
}

print("=" * 60)
print("CURRENT DEFAULT (model.py _TUNED_XGB_PARAMS)")
print("=" * 60)
_, report_current = train(df)
print(report_current)
continent_holdout_check(df)

if CANDIDATE_PARAMS:
    print("\n" + "=" * 60)
    print("CANDIDATE")
    print("=" * 60)
    _, report_candidate = train(df, params=CANDIDATE_PARAMS)
    print(report_candidate)
    continent_holdout_check(df, params=CANDIDATE_PARAMS)
else:
    print("\nno CANDIDATE_PARAMS set -- nothing to compare against. "
          "Fill it in with a real candidate before running this comparison.")

CURRENT DEFAULT (model.py _TUNED_XGB_PARAMS)
class balance (target = acidic):
acidic
0.0    0.694
1.0    0.306
              precision    recall  f1-score   support

         0.0      0.901     0.818     0.858     26652
         1.0      0.655     0.793     0.718     11604

    accuracy                          0.811     38256
   macro avg      0.778     0.806     0.788     38256
weighted avg      0.826     0.811     0.815     38256



c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



--- trained WITHOUT Europe, tested ON Europe (n=74127) ---
              precision    recall  f1-score   support

         0.0      0.802     0.723     0.760     48952
         1.0      0.548     0.652     0.595     25175

    accuracy                          0.699     74127
   macro avg      0.675     0.687     0.678     74127
weighted avg      0.715     0.699     0.704     74127


--- trained WITHOUT Africa, tested ON Africa (n=23248) ---
              precision    recall  f1-score   support

         0.0      0.843     0.739     0.788     17687
         1.0      0.404     0.564     0.471      5561

    accuracy                          0.697     23248
   macro avg      0.624     0.651     0.629     23248
weighted avg      0.738     0.697     0.712     23248


--- trained WITHOUT Northern America, tested ON Northern America (n=57115) ---
              precision    recall  f1-score   support

         0.0      0.801     0.754     0.777     41643
         1.0      0.428     0.497    

In [15]:
from agrishield.farmer_report import _load_crop_table, suggest_crops
crops = _load_crop_table()
print(len(crops), "crops loaded")
print(crops.head())

# fake a prediction to test suggest_crops without hitting GEE
fake_prediction = {"predicted": 1, "probability": {0: 0.3, 1: 0.7}}
print(suggest_crops(fake_prediction))

713 crops loaded
                     crop  ph_min  ph_max  \
0        Abbo rubber tree     4.5     5.0   
1      Abyssinian mustard     5.5     8.0   
2            Acacia Coral     5.0     7.0   
3    Acacia brachystachya     5.5     7.0   
4  Acanthosicyos horridus     6.0     7.0   

                                            category  
0                           fruits & nuts, materials  
1             forage/pasture, vegetables, cover crop  
2  forage/pasture, vegetables, materials, ornamen...  
3         forage/pasture, fruits & nuts, forest/wood  
4  fruits & nuts, vegetables, medicinals & aromat...  
['Abbo Rubber Tree', 'African Locust Bean', 'Italian Stone Pine', 'Mesua Ferrea', 'Muraina Grass']


In [16]:
"""Run this to confirm crop data is actually wired up end to end."""
from agrishield.model import load_model, predict_proba
from agrishield.inference import live_features, live_model_inputs
from agrishield.farmer_report import farmer_report, suggest_crops

model = load_model()

lat, lon = -37.91, 145.13  # swap in any real coordinate to test

inputs = live_model_inputs(lat, lon)
full = live_features(lat, lon)
result = predict_proba(model, inputs)

print("=== raw prediction ===")
print(result)

print("\n=== no crop given -- suggestion mode ===")
report_no_crop = farmer_report(lat, lon, model, inputs, result, full.get("weather_now"))
print(report_no_crop)
print("suggested crops for this soil:", suggest_crops(result))

print("\n=== farmer names a specific crop ===")
report_with_crop = farmer_report(lat, lon, model, inputs, result, full.get("weather_now"), crop="potato")
print(report_with_crop)

=== raw prediction ===
{'predicted': 1, 'probability': {0: 0.42549216747283936, 1: 0.5745078325271606}}

=== no crop given -- suggestion mode ===
{'location': {'latitude': -37.91, 'longitude': 145.13}, 'headline': 'Uncertain -- borderline result', 'verdict': 'uncertain', 'confidence_pct': 57.5, 'recommended_action': 'A physical soil test is recommended before making changes.', 'crop_note': None, 'oc_note': None, 'predicted_oc_gkg': None, 'color': '#F59E0B', 'context': {'ndvi': 0.6726943850517273, 'avg_temp_c': 14.4, 'avg_annual_rainfall_mm': 836, 'current_weather': 'overcast clouds'}, 'caveats': []}
suggested crops for this soil: ['Abbo Rubber Tree', 'African Locust Bean', 'Italian Stone Pine', 'Mesua Ferrea', 'Muraina Grass']

=== farmer names a specific crop ===
{'location': {'latitude': -37.91, 'longitude': 145.13}, 'headline': 'Uncertain -- borderline result', 'verdict': 'uncertain', 'confidence_pct': 57.5, 'recommended_action': 'A physical soil test is recommended before making ch

In [17]:
print(df["oc_gkg"].describe())
print("non-null:", df["oc_gkg"].notna().sum(), "/", len(df))

count    184605.000000
mean         37.642919
std          59.545796
min           0.000000
25%          10.333333
50%          18.200000
75%          35.700000
max         400.000000
Name: oc_gkg, dtype: float64
non-null: 184605 / 250578


In [18]:
# NEW CELL 1 -- train the regression model, separate from the classifier above
from agrishield.model import train_regression

oc_pipe, oc_metrics = train_regression(df)
print("organic carbon regression metrics:")
print(oc_metrics)

organic carbon regression metrics:
{'mae': 20.15850432409115, 'r2': 0.5241746780452143}


In [19]:
# NEW CELL 2 -- the same discipline we used for the classifier: don't trust
# the number above until you've checked it holds up region by region
from agrishield.model import continent_holdout_regression

continent_holdout_regression(df)

c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Europe: MAE=36.33, R2=-0.255 (n=63800)
Africa: MAE=12.10, R2=-0.046 (n=22466)
Asia: MAE=22.98, R2=-0.934 (n=6220)
South America: MAE=28.75, R2=0.030 (n=21794)
Northern America: MAE=51.05, R2=-0.036 (n=46711)
Oceania: MAE=17.95, R2=0.002 (n=23584)
skipping 'Antarctica': only 30 rows (< 100)


In [20]:
# NEW CELL 3 -- ONLY save if the metrics above look reasonable. Note the
# DIFFERENT filename -- this must not overwrite soil_risk_model.joblib
from agrishield.model import save_model
from agrishield.config import MODELS_DIR

save_model(oc_pipe, path=MODELS_DIR / "oc_model.joblib")

WindowsPath('D:/Jupyter Lab/Agrishield AI/models/oc_model.joblib')

In [21]:
# NEW CELL 4 -- end-to-end sanity check, same site you've used before
from agrishield.inference import live_model_inputs
from agrishield.model import predict_oc
from agrishield.farmer_report import farmer_report
from agrishield.config import EXAMPLE_SITE

inputs = live_model_inputs(EXAMPLE_SITE["latitude"], EXAMPLE_SITE["longitude"])
oc_value = predict_oc(oc_pipe, inputs)
print("predicted oc_gkg:", oc_value)

report = farmer_report(
    lat=EXAMPLE_SITE["latitude"], lon=EXAMPLE_SITE["longitude"],
    model=model, live_inputs=inputs, prediction=predict_proba(model, inputs),
    predicted_oc=oc_value,
)
print(report["oc_note"], "|", report["predicted_oc_gkg"])

predicted oc_gkg: 41.27195739746094
Organic carbon looks healthy -- likely less need for additional fertilizer. | 41.3
